## Zerobus setup - landing table and service principal grants

Run once per environment, before the first Zerobus producer run and before the Lakeflow pipeline picks up `ticks_bronze_zerobus.py`. Not part of any job.

Zerobus writes into an existing managed Delta table. It never evolves the table schema and can't write to a table that was dropped or recreated, so nothing here uses `CREATE OR REPLACE`. The tick itself stays a raw JSON string in `value`: new fields (for example `volume_24h`) need no DDL change.

In [0]:
%run ../config/config

In [0]:
sp_app_id = ZEROBUS_SP_APP_ID.strip()
full_table = BRONZE_ZEROBUS_TICKS

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {full_table} (
        value            STRING    COMMENT 'Raw tick JSON, same payload as the Event Hub producer',
        producer_id      STRING    NOT NULL COMMENT 'Producer run that sent the record',
        producer_seq     BIGINT    NOT NULL COMMENT 'Counter inside one producer run, a gap means a lost record',
        producer_sent_at TIMESTAMP NOT NULL COMMENT 'When the producer sent the record'
    )
    COMMENT 'Zerobus Ingest landing table for price ticks; may contain duplicates; read by the Lakeflow append flow into ticks_bronze'
""")

In [0]:
if not sp_app_id:
    raise ValueError("Set streaming.zerobus.sp_app_id in config/config.yml to the service principal application ID")

# write access to the landing table only, nothing on bronze, silver or gold
spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{sp_app_id}`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{BRONZE_SCHEMA} TO `{sp_app_id}`")
spark.sql(f"GRANT MODIFY, SELECT ON TABLE {full_table} TO `{sp_app_id}`")